# Lab 03 · Pandas for Data Manipulation and Cleaning
CS-3105 Machine Learning Lab · Lab Instructor: Engr. Muhammad Awais

CLO 6 (C-4 Analyze, PLO 2 Problem Analysis): Analyze real-world datasets by applying data manipulation and visualization techniques.

Data: `pakistan_used_cars.csv`, 24,973 real used-car ads from OLX Pakistan (Open Data Pakistan, Pakistan Used Cars, 2021). The ads are real and contain the problems real data has. Two kinds of formatting problem were added for practice (some prices written as text such as `Rs 1,430,000`, some city names in the wrong case or with extra spaces).

Rules: no Python loops over rows · label every output, e.g. `print("T4e:", ...)` · do not edit the CSV by hand · Runtime → Restart and run all before you submit · push this notebook to `ML-Lab-2024-SE-36/Lab03/Lab03.ipynb` by Thursday 5 November 2026, 11:59 PM (PKT).

Name: Sahadat Ali  Roll No: 2024-SE-36

## Warm-up (not graded): the class example from the lecture
Run this cell and repeat any lecture example on this small table before you start the tasks. `clean` is the tidy version, `messy` has the six kinds of problem.

In [1]:
import numpy as np
import pandas as pd
pd.set_option('display.width', 140)

clean = pd.DataFrame({
    "Roll": ["SE-01", "SE-02", "SE-03", "SE-04", "SE-05", "SE-06", "SE-07", "SE-08"],
    "Name": ["Ali", "Ayesha", "Bilal", "Fatima", "Hamza", "Sana", "Usman", "Zainab"],
    "City": ["Muzaffarabad", "Mirpur", "Bagh", "Muzaffarabad", "Rawalakot", "Mirpur", "Kotli", "Muzaffarabad"],
    "Maths": [78, 91, 45, 85, 62, 73, 38, 88],
    "Physics": [65, 88, 52, 79, 48, 81, 41, 92],
    "Attendance": [92, 97, 70, 88, 75, 90, 60, 95]})

messy = pd.DataFrame({
    "Roll": ["SE-01", "SE-02", "SE-03", "SE-04", "SE-05", "SE-06", "SE-07", "SE-04", "SE-08"],
    "Name": ["Ali", "Ayesha", "Bilal", "Fatima", "Hamza", "Sana", "Usman", "Fatima", "Zainab"],
    "City": [" Muzaffarabad", "Mirpur", "Bagh", "MUZAFFARABAD", "Rawalakot", "Mirpur", "Kotli", "MUZAFFARABAD", "Muzafarabad"],
    "Maths": [78, 91, np.nan, 85, 62, 73, 380, 85, 88],
    "Physics": [65, 88, 52, 79, 48, 105, 41, 79, 92],
    "Attendance": ["92%", "97", "70", "88", "75%", "90", "60", "88", "95"]})
print(clean)

    Roll    Name          City  Maths  Physics  Attendance
0  SE-01     Ali  Muzaffarabad     78       65          92
1  SE-02  Ayesha        Mirpur     91       88          97
2  SE-03   Bilal          Bagh     45       52          70
3  SE-04  Fatima  Muzaffarabad     85       79          88
4  SE-05   Hamza     Rawalakot     62       48          75
5  SE-06    Sana        Mirpur     73       81          90
6  SE-07   Usman         Kotli     38       41          60
7  SE-08  Zainab  Muzaffarabad     88       92          95


In [ ]:
# Try lecture examples here, e.g. clean[clean['Maths'] > 80]


### Setup
Run the Task 1 cell. If `pakistan_used_cars.csv` is not already available in the Colab session, it will ask you to upload it. Keep the exact filename `pakistan_used_cars.csv`.

In Google Colab, you can also upload the CSV using the folder icon on the left → Upload. Do not edit the CSV by hand.

## Task 1 · Load and first look
- **(a)** Read `pakistan_used_cars.csv` into `df` and print its shape.
- **(b)** Show the first 5 rows.
- **(c)** Print the column names.
- **(d)** Show the data type of every column.
- **(e)** Which column should be a number but is not? Show two of its values that explain why.

In [ ]:
# T1: Load and first look
import os
from pathlib import Path
import pandas as pd
import numpy as np
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", None)

# Find the required CSV; if it is missing in Colab, ask the user to upload it.
csv_path = Path("pakistan_used_cars.csv")
if not csv_path.exists():
    try:
        from google.colab import files
        print("Upload the file: pakistan_used_cars.csv")
        uploaded = files.upload()
        if csv_path.name not in uploaded and not csv_path.exists():
            csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
            if len(csv_files) == 1:
                os.replace(csv_files[0], csv_path)
            else:
                raise FileNotFoundError(
                    "Please upload exactly one CSV named pakistan_used_cars.csv."
                )
    except ImportError as exc:
        raise FileNotFoundError(
            "pakistan_used_cars.csv was not found. Upload it to the same folder "
            "as this notebook, then run this cell again."
        ) from exc

if not csv_path.exists():
    raise FileNotFoundError("Required file pakistan_used_cars.csv is still missing.")

df = pd.read_csv(csv_path)
required_columns = [
    "Brand", "Condition", "Fuel", "KMs Driven", "Model", "Price",
    "Registered City", "Transaction Type", "Year"
]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

# (a) shape
print("T1a:", df.shape)

# (b) first 5 rows
print("T1b:")
print(df.head())

# (c) column names
print("T1c:", list(df.columns))

# (d) data type of every column
print("T1d:")
print(df.dtypes)

# (e) a column that should be numeric but is text: Price
print("T1e: Price dtype =", df["Price"].dtype)
price_as_text = df["Price"].astype("string")
examples = price_as_text[price_as_text.str.contains("Rs", case=False, na=False)].head(2).tolist()
print("T1e: two values that explain why:", examples)


### Answer T1(e)
The **`Price`** column should be a number but its dtype is text (`object`, shown as `str` in newer pandas). Some ads store it as plain digits (`'2100000'`) and some with a prefix and commas (`'Rs 435,000'`, `'Rs 4,150,000'`), so pandas keeps the whole column as text.

## Task 2 · Retrieve data
- **(a)** The `Brand` column (first 5 values).
- **(b)** The columns `Brand`, `Model`, `Price` together (first 5 rows).
- **(c)** The first row, using `iloc`.
- **(d)** The `Model` in the row with index 100, using `loc`.
- **(e)** How many cars are registered in `"Islamabad"`?
- **(f)** How many **Honda** cars run on **Hybrid** fuel?
- **(g)** Cars from 2018 or newer, newest first (Brand, Model, Year, Price; first 10).
- **(h)** Try `df[df["Price"] > 1_000_000]`. What happens and why? Answer in a Markdown cell.

In [ ]:
# T2: Retrieve data

# (a) Brand column, first 5 values
print("T2a:")
print(df["Brand"].head())

# (b) Brand, Model, Price together
print("T2b:")
print(df[["Brand", "Model", "Price"]].head())

# (c) first row with iloc
print("T2c:")
print(df.iloc[0])

# (d) Model of the row with index 100 using loc
print("T2d:", df.loc[100, "Model"])

# (e) cars registered in Islamabad
print("T2e:", (df["Registered City"] == "Islamabad").sum())

# (f) Honda cars that run on Hybrid fuel
print("T2f:", ((df["Brand"] == "Honda") & (df["Fuel"] == "Hybrid")).sum())

# (g) cars from 2018 or newer, newest first (first 10)
print("T2g:")
print(df[df["Year"] >= 2018]
      .sort_values("Year", ascending=False)[["Brand", "Model", "Year", "Price"]]
      .head(10))

# (h) comparing the text Price column with a number
try:
    print(df[df["Price"] > 1_000_000])
except Exception as e:
    print("T2h: error ->", type(e).__name__, ":", e)

### Answer T2(h)
`df["Price"] > 1_000_000` raises a **TypeError**: Python cannot compare text (`str`) with a number (`int`). The `Price` column was read as text because some prices are written like `Rs 1,430,000`, which pandas cannot turn into numbers. So we first have to clean `Price` (remove `Rs` and commas, convert to `int64`, Task 3) before we can filter by price.

## Task 3 · Clean the data: the six types of problem
Fix the problems **in this order** (text first, then duplicates):
- **(a) Missing values:** count them per column. Drop ads with no `Brand` or no `Model`; fill missing `Registered City`, `Fuel` and `Transaction Type` with `"Unknown"`; leave `Year` and `KMs Driven` as NaN. Print how many rows were removed.
- **(b) Formatting:** remove extra spaces and fix the case of city names (`.str.strip().str.title()`); remove `"Rs"` and commas from `Price`.
- **(c) Data types:** convert `Price` to `int64` and `Year` to `Int64`.
- **(d) Inconsistent categories:** `"Sukkar"` → `"Sukkur"` and `"Corrolla Altis"` → `"Corolla Altis"`. Print the number of different cities before and after.
- **(e) Duplicates:** count exact duplicate ads, drop them, print the new row count.
- **(f) Invalid values:** set `KMs Driven` to NaN when it is below 100, above 1,000,000, or the placeholder 12345 / 123456. Drop ads with the fake prices 12,345,678 or 87,654,321, and Suzuki/Daihatsu ads above 5,000,000.
- **(g) Self-check:** your final shape must be **(20484, 9)**.

In [ ]:
# T3: Clean the data (order: missing -> formatting -> types -> categories -> duplicates -> invalid)

# (a) Missing values
print("T3a: missing values per column:")
print(df.isna().sum())
rows_before = len(df)
df = df.dropna(subset=["Brand", "Model"]).copy()
for col in ["Registered City", "Fuel", "Transaction Type"]:
    df[col] = df[col].fillna("Unknown")
# Year and KMs Driven are left as NaN on purpose.
print("T3a: rows removed (no Brand or no Model):", rows_before - len(df))

# (b) Formatting
df["Registered City"] = df["Registered City"].astype("string").str.strip().str.title()
price_text = (df["Price"].astype("string")
              .str.replace(r"(?i)rs\s*", "", regex=True)
              .str.replace(",", "", regex=False)
              .str.strip())

# (c) Data types
df["Price"] = pd.to_numeric(price_text, errors="raise").astype("int64")
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")
print("T3c: Price dtype =", df["Price"].dtype, "| Year dtype =", df["Year"].dtype)

# (d) Inconsistent categories
print("T3d: different cities BEFORE:", df["Registered City"].nunique())
df["Registered City"] = df["Registered City"].replace({"Sukkar": "Sukkur"})
df["Model"] = df["Model"].replace({"Corrolla Altis": "Corolla Altis"})
print("T3d: different cities AFTER:", df["Registered City"].nunique())

# (e) Duplicates
print("T3e: exact duplicate ads:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("T3e: rows after dropping duplicates:", len(df))

# (f) Invalid values
bad_km = ((df["KMs Driven"] < 100) | (df["KMs Driven"] > 1_000_000)
          | df["KMs Driven"].isin([12345, 123456]))
print("T3f: KMs Driven values set to NaN:", int(bad_km.sum()))
df.loc[bad_km, "KMs Driven"] = np.nan

fake_price = df["Price"].isin([12_345_678, 87_654_321])
too_high = df["Brand"].isin(["Suzuki", "Daihatsu"]) & (df["Price"] > 5_000_000)
print("T3f: ads dropped (fake prices):", int(fake_price.sum()),
      "| Suzuki/Daihatsu above 5,000,000:", int(too_high.sum()))
df = df.loc[~(fake_price | too_high)].copy()

# (g) Self-check
print("T3g: final shape:", df.shape)
assert df.shape == (20484, 9), (
    f"Expected cleaned shape (20484, 9), but got {df.shape}. "
    "Check that you uploaded the correct pakistan_used_cars.csv file."
)
print("T3g: self-check passed")


## Task 4 · Analyse
- **(a)** New column `Price_lakh` = price in lakh rupees (÷ 100,000).
- **(b)** New column `Age` = 2020 − `Year`.
- **(c)** The 5 most common brands.
- **(d)** The 5 cities with the most ads.
- **(e)** Median `Price_lakh` for each fuel type, highest first.
- **(f)** Median `Price_lakh` of a **Corolla GLI** for each year 2010–2018. How much value does it lose per year?
- **(g)** The 5 most expensive ads. Does one of them look wrong? Why?
- **(h)** Number of ads and median price for Cash vs Installment/Leasing. Interpret in 1–2 lines.

In [ ]:
# T4: Analyse

# (a) price in lakh rupees
df["Price_lakh"] = df["Price"] / 100_000
print("T4a:")
print(df[["Price", "Price_lakh"]].head())

# (b) age of the car (data is from 2020)
df["Age"] = 2020 - df["Year"]
print("T4b:")
print(df[["Year", "Age"]].head())

# (c) 5 most common brands
print("T4c:")
print(df["Brand"].value_counts().head(5))

# (d) 5 cities with the most ads
print("T4d:")
print(df["Registered City"].value_counts().head(5))

# (e) median Price_lakh for each fuel type, highest first
print("T4e:")
print(df.groupby("Fuel")["Price_lakh"].median().sort_values(ascending=False))

# (f) Corolla GLI median price per year 2010-2018 and yearly loss
gli = df[(df["Model"] == "Corolla GLI") & (df["Year"].between(2010, 2018))]
gli_median = gli.groupby("Year")["Price_lakh"].median()
print("T4f: median Price_lakh of Corolla GLI by year:")
print(gli_median)
print("T4f: change from one year to the next (lakh):")
print(gli_median.diff())
print("T4f: average value lost per year of age (lakh):", round(gli_median.diff().mean(), 2))

# (g) 5 most expensive ads
print("T4g:")
print(df.nlargest(5, "Price_lakh")[["Brand", "Model", "Year", "Condition", "KMs Driven", "Registered City", "Price_lakh"]])

# (h) Cash vs Installment/Leasing
print("T4h:")
print(df.groupby("Transaction Type")["Price_lakh"].agg(["count", "median"]))

### Answers T4
**T4(f):** The median price of a Corolla GLI rises steadily with the model year, from about 12.3 lakh (2010) to 23.2 lakh (2018). Equivalently, a Corolla GLI **loses about 1.37 lakh rupees for every extra year of age**.

**T4(g):** Yes, the 4th ad looks wrong: a **2007 Toyota Surf** priced at Rs 41,000,000 (410 lakh), marked "New" with only 3,222 km, in an ad with no city. A 12-year-old Surf should cost a small fraction of this, so the price is probably a typing mistake (extra zeros). The top ad (2017 Land Cruiser, 670 lakh, "New" but 32,222 km) is also doubtful, since the repeated digits in the km value suggest a made-up entry. These are rare, so the median is safe, but the mean would be pulled up.

**T4(h):** Cash ads are about 96% of the data (19,634 vs 850) and their median price is 6.9 lakh, while Installment/Leasing ads have a median of only 3.0 lakh. So installment/leasing ads are fewer and are mostly for cheaper cars. This shows an association only; it does not prove that paying in installments makes a car cheaper, and the listed price in such ads may be a down payment rather than the full price.

## Task 5 · Save your work
Save the cleaned DataFrame as `pakistan_used_cars_clean.csv` (Lab 04 uses it) and print its shape.

In [ ]:
# T5: Save the cleaned data
df.to_csv("pakistan_used_cars_clean.csv", index=False)
print("T5: saved, shape =", df.shape)

## Task 6 · Reflection (Markdown cell)
Write 3–5 sentences: which of the six cleaning problems would have given the most misleading answer if you had ignored it, and two things you learned about the used-car market in Pakistan.

The problem that would have given the most misleading answer is **duplicates**: 2,028 ads were exact copies, so without removing them the brand and city counts would be inflated by about 9% and nobody would get an error message. Wrong data types were nearly as bad, because 750 prices were stored as text like `Rs 435,000`; forcing them to numbers without cleaning would silently turn those ads into NaN, and leaving them as text makes every price question fail (Task 2h). Two things I learned about the Pakistani used-car market: it is dominated by a few brands and cities (Suzuki has about 42% of the ads and Karachi about 75%), and Hybrid cars have the highest median price (about 16.3 lakh), while a Corolla GLI loses about 1.37 lakh rupees for every extra year of age.